# 02 - Generalized ResUNet Reconstruction for Sparse-View CT

This notebook implements the learned post-processing stage of the sparse-view CT pipeline. To ensure maximum robustness and generalization, **we train a single, unified ResUNet model on all degraded images** (across all four sparse-view configurations: 180, 90, 60, and 45 views).

By exposing the network to different levels of streak artifacts during training, the ResUNet acts as a generalist artifact-reduction model that learns to reconstruct clean images from *any* under-sampled parallel-beam geometry setting.

The pipeline operates as follows:
1. Loads the preprocessed training/validation/testing datasets generated in notebook 00.
2. Computes the Filtered Backprojection (FBP) in memory for all four configurations to map noisy sinograms into image-domain proxy inputs containing streak artifacts.
3. Trains a **single generalized ResUNet** on the combination of all FBP degraded inputs against the clean ground-truth targets.
4. Evaluates the single trained network on a representative test slice, computing **PSNR** and **SSIM** metrics to demonstrate its generalization ability.
5. Plots and saves visual comparison panels along with absolute error maps.

## 1. Environment Setup and Imports

Mount the Google Drive, install `astra-toolbox`, and import PyTorch, NumPy, Matplotlib, and `IPPy` libraries.

In [ ]:
!pip install astra-toolbox

from google.colab import drive

drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed2"
WEIGHTS_DIR = PROJECT_ROOT / "weights" / "unet"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "unet"

for directory in [WEIGHTS_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import models, operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities.metrics import PSNR, SSIM

# Global Configuration
SEED = 42
IMAGE_SIZE = 256
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
ANGLE_COUNTS = (180, 90, 60, 45)  # The network will be trained on all of these

NUM_EPOCHS = 5  # Diagnostic run; increase to 20 only after the loss decreases reliably.
LEARNING_RATE = 1e-4
BASE_CHANNELS = 32
BATCH_SIZE = 8
MODEL_FINAL_ACTIVATION = None  # Avoid sigmoid saturation during training; clamp only for metrics/plots.
RESUME_TRAINING = True
PATIENCE = 3
RUN_OVERFIT_DEBUG = False
OVERFIT_DEBUG_VIEW = "60"
OVERFIT_DEBUG_SLICES = 16
OVERFIT_DEBUG_STEPS = 40
CHECKPOINT_PATH = WEIGHTS_DIR / "resunet_generalized_latest.pt"
BEST_CHECKPOINT_PATH = WEIGHTS_DIR / "resunet_generalized_best.pt"

torch.manual_seed(SEED)
np.random.seed(SEED)

device = utilities.get_device()
if str(device) == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Device used:", device)
if str(device) == "cuda":
    print("CUDA device:", torch.cuda.get_device_name(0))
print("Training on all view configurations:", ANGLE_COUNTS)
print("Model final activation:", MODEL_FINAL_ACTIVATION)
print("Processed data directory:", PROCESSED_DIR)
print("UNet weights directory:", WEIGHTS_DIR)
print("Latest checkpoint path:", CHECKPOINT_PATH)
print("Best checkpoint path:", BEST_CHECKPOINT_PATH)
print("UNet outputs directory:", OUTPUT_DIR)


## 2. Load the Preprocessed Data Contract

Identify patient files in the training, validation, and test splits and check the shapes of loaded sinograms.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

def collect_processed_patients(split_name: str) -> list[Path]:
    split_info = manifest["splits"][split_name]
    return [PROCESSED_DIR / record["path"] for record in split_info["patients"]]

processed_patients = {
    "train": collect_processed_patients("train"),
    "val": collect_processed_patients("val"),
    "test": collect_processed_patients("test"),
}

for split_name, patient_paths in processed_patients.items():
    num_images = manifest["splits"][split_name]["num_images"]
    print(f"{split_name} split: {num_images} images across {len(patient_paths)} patients")

example_payload = torch.load(processed_patients["train"][0], map_location="cpu")
print(f"Example clean shape: {tuple(example_payload['clean'].shape)}")
for views in ANGLE_COUNTS:
    print(f"Example sinogram ({views} views) shape: {tuple(example_payload['sinograms'][str(views)].shape)}")

## 3. FBP Proxy Setup

Set up a CT projector and FBP solver for each sparse-view configuration. The FBP output is used only as the degraded image-domain proxy input to the ResUNet.

In [ ]:
projectors = {
    str(n_views): operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
        force_cpu=True,
    )
    for n_views in ANGLE_COUNTS
}

fbp_solvers = {angle_key: solvers.FBP(projector) for angle_key, projector in projectors.items()}

def normalize_batch_per_image(batch: torch.Tensor) -> torch.Tensor:
    normalized = []
    for image in batch:
        normalized.append(normalize(image.unsqueeze(0)).clamp(0.0, 1.0).to(torch.float32))
    return torch.cat(normalized, dim=0)

@torch.no_grad()
def compute_fbp_proxy(sinogram: torch.Tensor, angle_key: str) -> torch.Tensor:
    fbp, _ = fbp_solvers[angle_key](sinogram.cpu(), x_true=None, starting_point=None)
    return normalize_batch_per_image(fbp.detach().cpu())


## 4. PyTorch Dataset and DataLoader

Set up the dataset class to load clean Mayo slices and their complete matching sinogram dictionary.

In [ ]:
class PatientSinogramDataset(Dataset):
    def __init__(self, patient_paths: list[Path]):
        self.patient_paths = list(patient_paths)

    def __len__(self):
        return len(self.patient_paths)

    def __getitem__(self, idx):
        patient_path = self.patient_paths[idx]
        payload = torch.load(patient_path, map_location="cpu")
        return {
            "sinograms": {key: val.detach().cpu().to(torch.float32) for key, val in payload["sinograms"].items()},
            "clean": payload["clean"].detach().cpu().to(torch.float32),
        }

def make_loader(patient_paths: list[Path], shuffle: bool) -> DataLoader:
    dataset = PatientSinogramDataset(patient_paths)
    return DataLoader(dataset, batch_size=None, shuffle=shuffle, num_workers=0)

def iter_slice_batches(patient_batch: dict, angle_key: str, batch_size: int = BATCH_SIZE):
    sinogram = torch.nan_to_num(patient_batch["sinograms"][angle_key].to(torch.float32), nan=0.0, posinf=0.0, neginf=0.0)
    clean = torch.nan_to_num(patient_batch["clean"].to(torch.float32), nan=0.0, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)
    for start in range(0, clean.shape[0], batch_size):
        end = start + batch_size
        yield sinogram[start:end], clean[start:end]

## 5. ResUNet Architecture

Define the IPPy UNet with residual down/up blocks. This keeps the U-Net encoder/decoder and skip-connection structure used in the course notes, while using IPPy's `ResidualConvBlock` implementation rather than the exact additive `ResidualDoubleConv` block shown in the slides.

In [ ]:
model = models.UNet(
    ch_in=1,
    ch_out=1,
    middle_ch=(BASE_CHANNELS, 2 * BASE_CHANNELS, 4 * BASE_CHANNELS, 8 * BASE_CHANNELS, 16 * BASE_CHANNELS),
    n_layers_per_block=2,
    down_layers=(
        "ResDownBlock",
        "ResDownBlock",
        "ResDownBlock",
        "ResDownBlock",
    ),
    up_layers=(
        "ResUpBlock",
        "ResUpBlock",
        "ResUpBlock",
        "ResUpBlock",
    ),
    n_heads=None,
    # Keep training logits unconstrained; output is clamped only for evaluation and display.
    final_activation=MODEL_FINAL_ACTIVATION,
).to(device)
model_device = next(model.parameters()).device
num_params = sum(param.numel() for param in model.parameters())
print(f"ResUNet successfully built. Total trainable parameters: {num_params / 1e6:.2f}M")
print("Model parameter device:", model_device)

## 6. Training the Single Generalized ResUNet Model

Train a **single ResUNet** by cycling through all four sparse-view geometries `(180, 90, 60, 45 views)` for each training batch. The sigmoid output is disabled during training to avoid saturation; predictions are clamped to `[0, 1]` only for metrics and visualization. Each epoch saves a latest checkpoint, while the lowest validation loss also saves a separate best checkpoint used for final evaluation.


In [ ]:
train_loader = make_loader(processed_patients["train"], shuffle=True)
val_loader = make_loader(processed_patients["val"], shuffle=False)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()
history = []
val_history = []
best_val_loss = float("inf")
epochs_without_improvement = 0
start_epoch = 0


def checkpoint_payload(epoch: int, train_loss: float, val_loss: float) -> dict:
    return {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
        "val_history": val_history,
        "config": {
            "ANGLE_COUNTS": ANGLE_COUNTS,
            "IMAGE_SIZE": IMAGE_SIZE,
            "BASE_CHANNELS": BASE_CHANNELS,
            "BATCH_SIZE": BATCH_SIZE,
            "LEARNING_RATE": LEARNING_RATE,
            "MODEL_FINAL_ACTIVATION": MODEL_FINAL_ACTIVATION,
            "PATIENCE": PATIENCE,
        },
    }


def load_compatible_checkpoint():
    global start_epoch, history, val_history, best_val_loss, epochs_without_improvement
    if RESUME_TRAINING and CHECKPOINT_PATH.exists():
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
        checkpoint_config = checkpoint.get("config", {})
        checkpoint_activation = checkpoint_config.get("MODEL_FINAL_ACTIVATION")
        has_activation_config = "MODEL_FINAL_ACTIVATION" in checkpoint_config
        if (not has_activation_config) or checkpoint_activation != MODEL_FINAL_ACTIVATION:
            print(
                "Checkpoint final activation is incompatible "
                f"({checkpoint_activation!r} != {MODEL_FINAL_ACTIVATION!r}). Starting from scratch."
            )
        else:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = int(checkpoint.get("epoch", 0))
            history = list(checkpoint.get("history", []))
            val_history = list(checkpoint.get("val_history", []))
            best_val_loss = float(checkpoint.get("best_val_loss", float("inf")))
            epochs_without_improvement = int(checkpoint.get("epochs_without_improvement", 0))
            print(f"Resuming training from checkpoint epoch {start_epoch}: {CHECKPOINT_PATH}")
    elif RESUME_TRAINING:
        print(f"RESUME_TRAINING=True, but no checkpoint found at {CHECKPOINT_PATH}. Starting from scratch.")
    else:
        print("RESUME_TRAINING=False. Starting training from scratch.")


def make_sanity_batch(angle_key: str = "60"):
    patient_batch = PatientSinogramDataset([processed_patients["train"][0]])[0]
    sinogram_batch, clean_batch = next(iter_slice_batches(patient_batch, angle_key, batch_size=BATCH_SIZE))
    fbp_input = compute_fbp_proxy(sinogram_batch, angle_key).to(device)
    clean_target = clean_batch.to(device)
    return fbp_input, clean_target


def print_sanity_check(label: str, fbp_input: torch.Tensor, clean_target: torch.Tensor):
    # These baselines detect collapse to a black image and check whether the network beats raw FBP.
    model.eval()
    with torch.no_grad():
        zero_loss = loss_fn(torch.zeros_like(clean_target), clean_target).item()
        fbp_loss = loss_fn(fbp_input, clean_target).item()
        model_loss = loss_fn(model(fbp_input), clean_target).item()
    print(f"{label} sanity MSE(zeros, clean): {zero_loss:.6f}")
    print(f"{label} sanity MSE(fbp_input, clean): {fbp_loss:.6f}")
    print(f"{label} sanity MSE(model(fbp_input), clean): {model_loss:.6f}")
    model.train()


def run_overfit_debug():
    print(
        f"Running manual overfit debug on view {OVERFIT_DEBUG_VIEW}, "
        f"{OVERFIT_DEBUG_SLICES} slices, {OVERFIT_DEBUG_STEPS} steps."
    )
    patient_batch = PatientSinogramDataset([processed_patients["train"][0]])[0]
    sinogram = patient_batch["sinograms"][OVERFIT_DEBUG_VIEW][:OVERFIT_DEBUG_SLICES]
    clean = patient_batch["clean"][:OVERFIT_DEBUG_SLICES].clamp(0.0, 1.0)
    fbp_input = compute_fbp_proxy(sinogram, OVERFIT_DEBUG_VIEW).to(device)
    clean_target = clean.to(device)

    model.train()
    initial_loss = None
    final_loss = None
    for step in range(OVERFIT_DEBUG_STEPS):
        optimizer.zero_grad(set_to_none=True)
        prediction = model(fbp_input)
        loss = loss_fn(prediction, clean_target)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss encountered during overfit debug!")
        loss.backward()
        optimizer.step()
        final_loss = loss.item()
        if initial_loss is None:
            initial_loss = final_loss
        if step == 0 or (step + 1) % 5 == 0 or step + 1 == OVERFIT_DEBUG_STEPS:
            print(f"Overfit debug step {step + 1:03d}/{OVERFIT_DEBUG_STEPS}: loss={final_loss:.6f}")

    if final_loss is not None and initial_loss is not None and final_loss >= 0.95 * initial_loss:
        print("WARNING: overfit debug loss barely decreased. Check pipeline, normalization, or model output scale.")


load_compatible_checkpoint()
sanity_fbp_input, sanity_clean_target = make_sanity_batch(angle_key="60")
print_sanity_check("Before training", sanity_fbp_input, sanity_clean_target)

if RUN_OVERFIT_DEBUG:
    run_overfit_debug()
    print("RUN_OVERFIT_DEBUG=True: skipping full multi-view training after the manual overfit check.")
elif start_epoch >= NUM_EPOCHS:
    print(f"Checkpoint is already at epoch {start_epoch}; NUM_EPOCHS={NUM_EPOCHS}. No training needed.")
else:
    print(f"Starting supervised training for Unified Generalized ResUNet model from epoch {start_epoch + 1} to {NUM_EPOCHS}...")

    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        num_samples = 0
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [train]")

        for patient_batch in progress:
            # Loop through all 4 angle geometries to expose the network to all levels of degradation.
            for angle_key in [str(n) for n in ANGLE_COUNTS]:
                for sinogram_batch, clean_batch in iter_slice_batches(patient_batch, angle_key):
                    fbp_input = compute_fbp_proxy(sinogram_batch, angle_key).to(device)
                    clean_target = clean_batch.to(device)

                    optimizer.zero_grad(set_to_none=True)
                    prediction = model(fbp_input)
                    loss = loss_fn(prediction, clean_target)

                    if not torch.isfinite(loss):
                        raise RuntimeError("Non-finite loss encountered during training!")

                    loss.backward()
                    optimizer.step()

                    batch_size = clean_target.shape[0]
                    running_loss += loss.item() * batch_size
                    num_samples += batch_size
                    progress.set_postfix(train_loss=running_loss / num_samples)

        epoch_loss = running_loss / num_samples
        history.append(epoch_loss)

        model.eval()
        val_running_loss = 0.0
        val_num_samples = 0
        with torch.no_grad():
            for patient_batch in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [val]", leave=False):
                for angle_key in [str(n) for n in ANGLE_COUNTS]:
                    for sinogram_batch, clean_batch in iter_slice_batches(patient_batch, angle_key):
                        fbp_input = compute_fbp_proxy(sinogram_batch, angle_key).to(device)
                        clean_target = clean_batch.to(device)
                        prediction = model(fbp_input)
                        val_loss = loss_fn(prediction, clean_target)

                        batch_size = clean_target.shape[0]
                        val_running_loss += val_loss.item() * batch_size
                        val_num_samples += batch_size

        epoch_val_loss = val_running_loss / val_num_samples
        val_history.append(epoch_val_loss)
        improved = epoch_val_loss < best_val_loss - 1e-6
        if improved:
            best_val_loss = epoch_val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(f"Epoch {epoch + 1}/{NUM_EPOCHS} | train loss: {epoch_loss:.6f} | val loss: {epoch_val_loss:.6f}")

        payload = checkpoint_payload(epoch + 1, epoch_loss, epoch_val_loss)
        torch.save(payload, CHECKPOINT_PATH)
        print(f"Saved latest checkpoint after epoch {epoch + 1}: {CHECKPOINT_PATH}")

        if improved:
            torch.save(payload, BEST_CHECKPOINT_PATH)
            print(f"Saved best checkpoint after epoch {epoch + 1}: {BEST_CHECKPOINT_PATH}")

        if epoch == start_epoch:
            print_sanity_check("After first epoch", sanity_fbp_input, sanity_clean_target)

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping triggered after {PATIENCE} epochs without validation improvement.")
            break

print(f"Latest training checkpoint available at: {CHECKPOINT_PATH}")
print(f"Best validation checkpoint available at: {BEST_CHECKPOINT_PATH}")


## 7. Unified Training Loss Curve

Plot the mean squared error loss across epochs to verify the convergence behavior of our model. The curve is saved inside the `/outputs/unet/` folder.

In [ ]:
training_curve_path = OUTPUT_DIR / "resunet_generalized_training_curve.png"

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(range(1, len(history) + 1), history, marker="o", color="dodgerblue", label="Train")
ax.plot(range(1, len(val_history) + 1), val_history, marker="o", color="tab:orange", label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Generalized ResUNet Training and Validation Loss")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(training_curve_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved training curve figure:", training_curve_path)

## 8. Generalization Test Set Evaluation and Metrics

Evaluate the best-validation ResUNet checkpoint on a representative slice of the test patient. Predictions are clamped to `[0, 1]` only at evaluation time before computing **PSNR** and **SSIM**.


In [ ]:
# Load best validation model weights. Fall back to latest only if no best checkpoint exists.
evaluation_checkpoint_path = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH.exists() else CHECKPOINT_PATH
checkpoint = torch.load(evaluation_checkpoint_path, map_location=device)
checkpoint_config = checkpoint.get("config", {})
checkpoint_activation = checkpoint_config.get("MODEL_FINAL_ACTIVATION")
has_activation_config = "MODEL_FINAL_ACTIVATION" in checkpoint_config
if (not has_activation_config) or checkpoint_activation != MODEL_FINAL_ACTIVATION:
    raise RuntimeError(
        "Evaluation checkpoint is incompatible with the current model final activation "
        f"({checkpoint_activation!r} != {MODEL_FINAL_ACTIVATION!r}). Re-run training first."
    )
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded ResUNet checkpoint from epoch {checkpoint.get('epoch', 'unknown')}: {evaluation_checkpoint_path}")

# Load the first test patient
test_split = manifest["splits"]["test"]
patient_record = test_split["patients"][0]
patient_path = PROCESSED_DIR / patient_record["path"]
payload = torch.load(patient_path, map_location="cpu")

clean_images = payload["clean"].to(torch.float32)
sinograms = {key: val.to(torch.float32) for key, val in payload["sinograms"].items()}
metadata = payload["metadata"]
patient_id = metadata["patient_id"]

print(f"Loaded Test Patient ID: {patient_id}")

# Select the central slice for quick evaluation
slice_idx = clean_images.shape[0] // 2
print(f"Evaluating on central test slice index: {slice_idx}")

x_true = clean_images[slice_idx : slice_idx + 1].to(device)

evaluation_results = {}
comparison_examples = {}

def compute_slice_metrics(prediction: torch.Tensor, clean: torch.Tensor) -> dict[str, float]:
    pred_cpu = torch.clamp(prediction.detach().cpu(), 0.0, 1.0)
    clean_cpu = torch.clamp(clean.detach().cpu(), 0.0, 1.0)
    return {
        "psnr": float(PSNR(pred_cpu, clean_cpu)),
        "ssim": float(SSIM(pred_cpu, clean_cpu)),
    }

for angle_key in [str(n) for n in ANGLE_COUNTS]:
    y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(device)
    fbp_input = compute_fbp_proxy(y_delta, angle_key).to(device)

    with torch.no_grad():
        # Clamp only for physical image-domain metrics and display; training uses unconstrained logits.
        prediction = torch.clamp(model(fbp_input), 0.0, 1.0)

    resunet_metrics = compute_slice_metrics(prediction, x_true)

    evaluation_results[angle_key] = {
        "resunet_psnr": resunet_metrics["psnr"],
        "resunet_ssim": resunet_metrics["ssim"],
    }

    comparison_examples[angle_key] = {
        "fbp": fbp_input.detach().cpu(),
        "resunet": prediction.detach().cpu(),
        "clean": x_true.detach().cpu(),
    }

    print(f"\n{angle_key} views:")
    print(f"  ResUNet PSNR: {resunet_metrics['psnr']:.2f} dB | SSIM: {resunet_metrics['ssim']:.4f}")



## 9. Visual Inspection and Absolute Error Maps

Plot the Ground Truth image, the FBP proxy input, the final ResUNet reconstruction output, and the ResUNet absolute error map. The FBP image is shown only as the degraded input given to the network.

In [ ]:
for angle_key in [str(n) for n in ANGLE_COUNTS]:
    example = comparison_examples[angle_key]
    gt_img = torch.clamp(example["clean"][0, 0], 0.0, 1.0).numpy()
    fbp_img = torch.clamp(example["fbp"][0, 0], 0.0, 1.0).numpy()
    resunet_img = torch.clamp(example["resunet"][0, 0], 0.0, 1.0).numpy()

    resunet_error = np.abs(resunet_img - gt_img)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    fig.suptitle(f"ResUNet Reconstruction Panel - {angle_key} views - Slice {slice_idx}", fontsize=14)

    axes[0].imshow(gt_img, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(fbp_img, cmap="gray", vmin=0.0, vmax=1.0)
    axes[1].set_title("FBP Proxy Input")
    axes[1].axis("off")

    axes[2].imshow(resunet_img, cmap="gray", vmin=0.0, vmax=1.0)
    axes[2].set_title(f"ResUNet Reconstruction\nPSNR: {evaluation_results[angle_key]['resunet_psnr']:.2f} dB | SSIM: {evaluation_results[angle_key]['resunet_ssim']:.4f}")
    axes[2].axis("off")

    im_err = axes[3].imshow(resunet_error, cmap="magma")
    axes[3].set_title("Absolute Error |ResUNet - GT|")
    axes[3].axis("off")
    fig.colorbar(im_err, ax=axes[3], shrink=0.8)

    output_fig_path = OUTPUT_DIR / f"resunet_reconstruction_panel_{angle_key}.png"
    fig.savefig(output_fig_path, dpi=150)
    plt.show()
    plt.close(fig)

    print(f"Saved ResUNet reconstruction panel for {angle_key} views:", output_fig_path)

## 10. Quantitative Results Summary Table

Print a consolidated ASCII table summarizing the required PSNR and SSIM performance of the trained Generalized ResUNet across all tested sparse-view configurations.

In [ ]:
print("="*55)
print(f"{ 'VIEWS':<10} | { 'METHOD':<16} | { 'PSNR (dB)':<12} | { 'SSIM':<8}")
print("="*55)
for angle_key in [str(n) for n in ANGLE_COUNTS]:
    res = evaluation_results[angle_key]
    print(f"{ angle_key:<10} | { 'ResUNet':<16} | { res['resunet_psnr']:<12.4f} | { res['resunet_ssim']:<8.4f}")
print("="*55)

## 11. Quantitative Plot

Save the required PSNR/SSIM plot for the ResUNet method across all sparse-view configurations.

In [ ]:
view_labels = [str(n) for n in ANGLE_COUNTS]
psnr_values = [evaluation_results[key]["resunet_psnr"] for key in view_labels]
ssim_values = [evaluation_results[key]["resunet_ssim"] for key in view_labels]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
fig.suptitle(f"ResUNet Quantitative Results - Test Slice {slice_idx}", fontsize=13)

axes[0].plot(view_labels, psnr_values, marker="o", color="tab:green")
axes[0].set_xlabel("Number of views")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR")

axes[1].plot(view_labels, ssim_values, marker="o", color="tab:green")
axes[1].set_ylim(0.0, 1.0)
axes[1].set_xlabel("Number of views")
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM")

plot_path = OUTPUT_DIR / "resunet_quantitative_results_all_views.png"
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved ResUNet quantitative plot:", plot_path)